# Nguyen et al. 2022 enantiomeric mixture -- analysis

Loads the saved samples produced by `fit.py` for both datasets (run it first, e.g.
`python examples/nguyen_2022_enantiomeric_mixture/fit.py --quick`) and reproduces the
S3-Table 95% credible intervals, the Baum_59 composition posterior, and posterior-predictive
checks. `Fokkens_1d` uses the racemic-mixture (RM) model and `Baum_59` the enantiomeric-mixture
(EM) model. No sampling happens in this notebook.

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")

EXAMPLE_NAME = "nguyen_2022_enantiomeric_mixture"
cwd = Path.cwd()
repo = cwd if cwd.name == "BayesianBinding" else next(
    (p for p in [cwd, *cwd.parents] if p.name == "BayesianBinding"), None
)
if repo is None:
    raise RuntimeError("Run this notebook from inside the BayesianBinding repository.")
sys.path.insert(0, str(repo / "src"))
sys.path.insert(0, str(repo / "examples"))
EXAMPLE_DIR = repo / "examples" / EXAMPLE_NAME
RESULTS_DIR = EXAMPLE_DIR / "results"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import _common

Load both saved result bundles and the dataset metadata.

In [ ]:
import json
from bayesian_binding.data import load_dat

data_dir = repo / "data" / "nguyen_2022_racemic_mixture"
metadata = json.loads((data_dir / "metadata.json").read_text())["datasets"]

fokkens_samples, fokkens_summary, fokkens_meta = _common.load_results(RESULTS_DIR / "fokkens_1d_rm")
baum_samples, baum_summary, baum_meta = _common.load_results(RESULTS_DIR / "baum_59_em")
print("Fokkens_1d (RM):", fokkens_meta["mode"], fokkens_meta["mcmc"])
print("Baum_59 (EM):   ", baum_meta["mode"], baum_meta["mcmc"])


def load_experiment(name):
    m = metadata[name]
    return load_dat(
        data_dir / m["file"],
        cell_concentration_mM=m["cell_concentration_mM"],
        syringe_concentration_mM=m["syringe_concentration_mM"],
        cell_volume_mL=m["cell_volume_mL"],
        temperature_k=m["temperature_K"],
        name=name,
    )

## Fokkens_1d (racemic-mixture model)
Computed 95% BCIs versus S3 Table; all four thermodynamic parameters should agree within ~0.1 kcal/mol.

In [ ]:
from bayesian_binding.regression import (
    NGUYEN_2022_S3_BCI,
    nguyen_2022_regression_rows,
    summarize_racemic_mixture_posterior,
)

fokkens_reg = summarize_racemic_mixture_posterior(fokkens_samples)
pd.DataFrame(nguyen_2022_regression_rows(fokkens_reg, "Fokkens_1d", "racemic_mixture"))

## Baum_59 (enantiomeric-mixture model)
The headline result is the composition `rho`, sharply peaked near 0.15 (clearly non-racemic). With concentrations unknown the enthalpies are broad; only `rho` is reproduced at full precision.

In [ ]:
baum_reg = summarize_racemic_mixture_posterior(baum_samples)
pd.DataFrame(nguyen_2022_regression_rows(baum_reg, "Baum_59", "enantiomeric_mixture"))

In [ ]:
rho = np.asarray(baum_samples["rho"])
fig, ax = plt.subplots(figsize=(6.0, 3.6))
ax.hist(rho, bins=60, color="#6baed6", edgecolor="white")
lo, hi = NGUYEN_2022_S3_BCI["Baum_59"]["enantiomeric_mixture"]["rho"]
ax.axvline(lo, color="#d94801", ls="--", lw=1.2)
ax.axvline(hi, color="#d94801", ls="--", lw=1.2, label="S3 Table 95% BCI")
ax.axvline(0.5, color="0.4", ls=":", lw=1.2, label="racemic (rho = 0.5)")
ax.set_xlabel("rho (mole fraction of ligand 1)"); ax.set_ylabel("count")
ax.set_title("Baum_59 EM composition posterior"); ax.legend(frameon=False); fig.tight_layout();

## Posterior predictive checks

In [ ]:
def predictive(samples_, name, title):
    experiment = load_experiment(name)
    q = np.asarray(samples_["q_model"])
    mean = q.mean(axis=0)
    interval = np.quantile(q, [0.025, 0.975], axis=0)
    x = np.arange(1, experiment.n_injections + 1)
    fig, ax = plt.subplots(figsize=(6.2, 3.8))
    ax.fill_between(x, interval[0], interval[1], color="#9ecae1", alpha=0.45, label="95% posterior predictive")
    ax.plot(x, mean, color="#08519c", lw=2.0, label="posterior predictive mean")
    ax.scatter(x, experiment.heats_microcalorie, color="black", s=24, zorder=3, label="observed")
    ax.axhline(0.0, color="0.82", lw=0.8)
    ax.set_xlabel("injection"); ax.set_ylabel("heat (microcal)"); ax.set_title(title)
    ax.legend(frameon=False); fig.tight_layout()

predictive(fokkens_samples, "Fokkens_1d", "Fokkens_1d RM posterior predictive")
predictive(baum_samples, "Baum_59", "Baum_59 EM posterior predictive");